# Facilitator-Member Difference Identification

## Creating the csv file for the following:

`Name-conference-year`

**Also for the following:**

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

## Regression Analysis

Figuring out the question: 

**By analyzing at the individual level, do facilitators act differently than team members?**

Also, considering again, the classification from above:

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

# Running the features alone (without existing knowledge)

Determining if people are (or not) facilitator, funded-team, on-team.

**Probably would just run cross-validation on this.**

## Fine-tuning stuff (09/29/2025)

### floating points

In [4]:
import json, pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict, Counter

# ========= CONFIG =========
DATA_DIR   = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")  # root with 2021MZT, 2022SLU, ...
OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_PERSON_SESSION = True  # set False to skip writing the audit csv

# ========= HELPERS =========
def _load_json(fp: Path):
    with open(fp, "r") as f:
        return json.load(f)

def extract_person_metrics_from_session_data(conf_path: Path, session_id: str):
    """
    Compute per-person p_* metrics for a single session from /session_data/<session_id>.json.
    Returns dict: person -> {p_*: value}
    """
    f = conf_path / "session_data" / f"{session_id}.json"
    if not f.exists():
        return {}

    data = _load_json(f)
    rows = data.get("all_data", [])
    by_person = defaultdict(lambda: Counter())
    for r in rows:
        speaker = r.get("speaker")
        if not speaker:
            continue
        dur = r.get("speaking_duration", 0) or 0
        by_person[speaker]["p_speaking_duration_sec"] += float(dur)
        by_person[speaker]["p_turns"] += 1

        if str(r.get("interuption","")).strip().lower() == "yes":
            by_person[speaker]["p_interruptions_made"] += 1
        if str(r.get("overlap","")).strip().lower() == "yes":
            by_person[speaker]["p_overlaps"] += 1
        if str(r.get("screenshare","")).strip().lower() == "yes":
            by_person[speaker]["p_screenshare_segments"] += 1

        by_person[speaker]["p_smile_self_total"]  += float(r.get("smile_self", 0) or 0)
        by_person[speaker]["p_smile_other_total"] += float(r.get("smile_other", 0) or 0)
        by_person[speaker]["p_nods_received"]     += float(r.get("nods_others", 0) or 0)

        # OPTIONAL: collect annotation counts/scores at utterance level if present
        ann = r.get("annotations") or {}
        for cat, info in ann.items():
            # normalize category name to snake-ish
            key = (
                cat.lower()
                   .replace("&","and")
                   .replace("/","_")
                   .replace(" ","_")
                   .replace("-", "_")
            )
            score = info.get("score", None)
            if score is not None:
                by_person[speaker][f"ann_{key}_count"] += 1
                by_person[speaker][f"ann_{key}_sum_score"] += float(score)

    # return dict of dicts
    return {p: dict(cnt) for p, cnt in by_person.items()}

def load_conference(conf_path: Path):
    conf_name = conf_path.name  # e.g., 2021MZT
    year = int(conf_name[:4])
    conference = conf_name[4:]

    session_outcomes = _load_json(conf_path / f"{conf_name}_session_outcomes.json")

    # features_*.json (session-level context; we’ll prefix ctx_*)
    features = {}
    for fp in conf_path.glob("features_*.json"):
        sid = fp.stem.replace("features_", "")
        features[sid] = _load_json(fp)

    return year, conference, session_outcomes, features

# ========= BUILD PERSON-SESSION =========
all_ps = []
for conf_path in sorted(DATA_DIR.iterdir()):
    if not conf_path.is_dir():
        continue
    conf_name = conf_path.name
    core = conf_path / f"{conf_name}_session_outcomes.json"
    if not core.exists():
        continue

    year, conference, session_outcomes, features = load_conference(conf_path)
    special = {"missing_names", "people_not_in_any_team"}
    sessions = [sid for sid in session_outcomes.keys() if sid not in special]

    for sid in sessions:
        so = session_outcomes[sid]
        facilitators = set(so.get("facilitators", []) or [])
        speakers     = set(so.get("all_speakers", []) or [])

        # team members from teams{}
        members = set()
        for _, tinfo in (so.get("teams", {}) or {}).items():
            for m in tinfo.get("members", []) or []:
                members.add(m)

        people = sorted(facilitators | speakers | members)
        ctx = {f"ctx_{k}": v for k, v in (features.get(sid, {}) or {}).items()}
        pmet = extract_person_metrics_from_session_data(conf_path, sid)

        for person in people:
            if person in facilitators:
                role_in_session = "facilitator"
            elif person in members:
                role_in_session = "member"
            elif person in speakers:
                role_in_session = "participant"
            else:
                role_in_session = "unknown"

            row = {
                "person_name": person,
                "conference": conference,
                "year": year,
                "session_id": sid,
                "role_in_session": role_in_session,
                **ctx
            }
            row.update(pmet.get(person, {}))
            all_ps.append(row)

ps_df = pd.DataFrame(all_ps)

# write person-session (audit)
if WRITE_PERSON_SESSION and not ps_df.empty:
    ps_out = OUTPUT_DIR / "ALL_person_session.csv"
    ps_df.to_csv(ps_out, index=False)
    print(f"Wrote person-session audit: {ps_out}  ({len(ps_df)} rows)")

# ========= AGGREGATE → PERSON-YEAR (SUM COUNTS; WEIGHTED MEANS FOR ann_*_mean_score) =========
if ps_df.empty:
    raise SystemExit("No person-session rows. Check DATA_DIR structure/files.")

keys = ["person_name","conference","year"]

# identify columns
p_cols = [c for c in ps_df.columns if c.startswith("p_")]
ctx_cols = [c for c in ps_df.columns if c.startswith("ctx_")]
ann_count_cols = [c for c in ps_df.columns if c.startswith("ann_") and c.endswith("_count")]
ann_sum_cols   = [c for c in ps_df.columns if c.startswith("ann_") and c.endswith("_sum_score")]

# base aggregations
agg_map = {}
# p_* sum (counts/segments) and duration sum
for c in p_cols:
    agg_map[c] = "sum"
# ctx_* mean across sessions
for c in ctx_cols:
    agg_map[c] = "mean"
# annotations: sum counts and sum_scores
for c in ann_count_cols + ann_sum_cols:
    agg_map[c] = "sum"

py_base = ps_df.groupby(keys, as_index=False).agg(agg_map)

# add sessions_total (unique sessions attended)
sesh_counts = (
    ps_df.groupby(keys, as_index=False)["session_id"]
         .nunique()
         .rename(columns={"session_id":"sessions_total"})
)
py = py_base.merge(sesh_counts, on=keys, how="left")

# derive role tallies per person-year from person-session
role_tallies = (
    ps_df.assign(one=1)
         .pivot_table(index=keys, columns="role_in_session", values="one", aggfunc="sum", fill_value=0)
         .reset_index()
         .rename(columns={"facilitator":"facilitator", "member":"member", "participant":"participant", "unknown":"unknown"})
)
# ensure all role columns exist
for col in ["facilitator","member","participant","unknown"]:
    if col not in role_tallies.columns:
        role_tallies[col] = 0
py = py.merge(role_tallies, on=keys, how="left")

# weighted means for each annotation mean_score = sum_score / count
ann_categories = set([c.replace("ann_","").replace("_count","") for c in ann_count_cols])
for cat in sorted(ann_categories):
    cnt_col = f"ann_{cat}_count"
    sum_col = f"ann_{cat}_sum_score"
    mean_col = f"ann_{cat}_mean_score"
    if cnt_col in py.columns and sum_col in py.columns:
        py[mean_col] = np.where(py[cnt_col].fillna(0) > 0,
                                pd.to_numeric(py[sum_col], errors="coerce") / pd.to_numeric(py[cnt_col], errors="coerce"),
                                np.nan)

# ===== dtype enforcement: ints for counts; floats for durations/scores =====
# counts (discrete)
count_like = [
    "p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "sessions_total","facilitator","member","participant","unknown"
] + ann_count_cols

for c in count_like:
    if c in py.columns:
        py[c] = pd.to_numeric(py[c], errors="coerce").fillna(0).round().astype(int)

# sums of scores are numeric; they’ll typically be floats but are sums of discrete scores (keep as int if integral)
for c in ann_sum_cols:
    if c in py.columns:
        s = pd.to_numeric(py[c], errors="coerce").fillna(0)
        py[c] = np.where(np.isclose(s, np.round(s)), s.round().astype(int), s)  # keep int if whole, else float

# speaking duration is continuous; keep float (round for readability)
if "p_speaking_duration_sec" in py.columns:
    py["p_speaking_duration_sec"] = pd.to_numeric(py["p_speaking_duration_sec"], errors="coerce").round(3)

# ===== OPTIONAL: derive role flags =====
py["role_primary"] = py[["facilitator","member","participant","unknown"]].idxmax(axis=1)
py["role_facilitator"]   = (py["facilitator"]   > 0).astype(int)
py["role_member"]        = (py["member"]        > 0).astype(int)
py["role_participant"]   = (py["participant"]   > 0).astype(int)
py["role_nonfacilitator"]= (py["role_facilitator"] == 0).astype(int)

# If you have team files to compute on_team / funded, merge here (skipped because not loaded in this block)

# write person-year
py_out = OUTPUT_DIR / "ALL_person_year_FIXED.csv"
py.sort_values(["person_name","year","conference"]).to_csv(py_out, index=False)
print(f"Wrote person-year: {py_out}  ({len(py)} rows)")

# ========= AGGREGATE → PERSON (across all years) =========
sum_cols_person = [
    "p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "sessions_total","facilitator","member","participant","unknown"
] + ann_count_cols + ann_sum_cols

mean_cols_person = [c for c in py.columns if c.startswith(("ctx_"))]  # context means (optional)
mean_cols_person += [c for c in py.columns if c.endswith("_mean_score")]  # annotation means

agg_person = {}
for c in sum_cols_person:
    if c in py.columns: agg_person[c] = "sum"
for c in mean_cols_person:
    if c in py.columns: agg_person[c] = "mean"

person = py.groupby("person_name", as_index=False).agg(agg_person)

# enforce ints again for counts
for c in sum_cols_person:
    if c in person.columns:
        person[c] = pd.to_numeric(person[c], errors="coerce").fillna(0).round().astype(int)

# keep durations and mean scores as floats (rounded)
for c in [x for x in person.columns if x.endswith("_mean_score") or x.startswith(("ctx_","p_speaking_duration_sec"))]:
    person[c] = pd.to_numeric(person[c], errors="coerce").round(3)

person_out = OUTPUT_DIR / "all_data_df-agg-ppl_FIXED.csv"
person.sort_values(["person_name"]).to_csv(person_out, index=False)
print(f"Wrote person-level: {person_out}  ({len(person)} rows)")

# ========= QUICK SANITY: show any columns that are float but expected int ========
def check_int_columns(df, cols, label):
    bad = []
    for c in cols:
        if c in df.columns and not pd.api.types.is_integer_dtype(df[c]):
            bad.append(c)
    if bad:
        print(f"[WARN] {label} columns not integer as expected:", bad)
    else:
        print(f"[OK] All {label} columns are integer.")

int_cols_py = ["p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
               "sessions_total","facilitator","member","participant","unknown"] + ann_count_cols
check_int_columns(py, int_cols_py, "person-year count-like")

int_cols_person = [c for c in int_cols_py + ann_sum_cols if c in person.columns]
check_int_columns(person, int_cols_person, "person-level count-like")

Wrote person-session audit: /Users/maxchalekson/Desktop/outputs/ALL_person_session.csv  (2073 rows)
Wrote person-year: /Users/maxchalekson/Desktop/outputs/ALL_person_year_FIXED.csv  (790 rows)
Wrote person-level: /Users/maxchalekson/Desktop/outputs/all_data_df-agg-ppl_FIXED.csv  (670 rows)
[OK] All person-year count-like columns are integer.
[OK] All person-level count-like columns are integer.


### fixing the names

In [7]:
import re, unicodedata
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

# ========= PATHS =========
IN_DIR = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/")
PS_IN  = IN_DIR / "ALL_person_session.csv"
PY_IN  = IN_DIR / "ALL_person_year_FIXED.csv"
P_IN   = IN_DIR / "all_data_df-agg-ppl_FIXED.csv"

OUT_DIR = IN_DIR
MAP_OUT = OUT_DIR / "name_clean_map.csv"
PS_OUT  = OUT_DIR / "ALL_person_session_CLEANED.csv"
PY_OUT  = OUT_DIR / "ALL_person_year_FIXED_CLEANED.csv"
P_OUT   = OUT_DIR / "all_data_df-agg-ppl_FIXED_CLEANED.csv"

# ========= LOAD =========
ps = pd.read_csv(PS_IN)
py = pd.read_csv(PY_IN)
p  = pd.read_csv(P_IN)

# ========= NAME NORMALIZATION =========
DEGREE_SUFFIXES = [
    r"\bph\.?d\.?\b", r"\bmd\b", r"\bmd-ph\.?d\.?\b", r"\bscd\b", r"\bmsc\b", r"\bms\b",
    r"\bba\b", r"\bma\b", r"\bmba\b", r"\bdvm\b", r"\bdds\b"
]
ORG_DELIMS = [r"\s*-\s*", r"\s*—\s*", r"\s*–\s*", r"\s*\|\s*", r"\s*@\s*"]
TRAILERS   = [r"\(.*?\)", r"\[.*?\]", r"\{.*?\}"]

# If you know any manual fixes, put them here (raw or cleaned key -> canonical value)
alias_map = {
    # "Max Chalekson - UCLA": "Max Chalekson",
    # "Evey Huang, PhD": "Evey Huang",
}

def strip_accents(text: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))

def basic_name_clean(s: str) -> str:
    if not isinstance(s, str) or not s.strip():
        return ""
    x = s.strip()

    # remove emails
    x = re.sub(r"\b\S+@\S+\.\S+\b", " ", x)
    # drop anything in (), [], {}
    for patt in TRAILERS:
        x = re.sub(patt, " ", x)
    # drop degree suffixes
    for deg in DEGREE_SUFFIXES:
        x = re.sub(deg, " ", x, flags=re.IGNORECASE)
    # split off affiliation/org
    for delim in ORG_DELIMS:
        x = re.split(delim, x)[0]

    # remove titles
    x = re.sub(r"^\s*(dr|prof|mr|ms|mrs)\.?\s+", " ", x, flags=re.IGNORECASE)

    # tidy punctuation/whitespace
    x = re.sub(r"[^\w\s'.-]", " ", x)            # keep letters/digits/space/apostrophe/dot/hyphen
    x = re.sub(r"\s+", " ", x).strip()

    # accents -> ascii, title-case words
    x = strip_accents(x)
    x = " ".join(w.capitalize() for w in x.split())

    # length guard
    return x[:120].strip()

def final_clean(raw: str) -> str:
    if not isinstance(raw, str): return ""
    if raw in alias_map: return alias_map[raw]
    cleaned = basic_name_clean(raw)
    return alias_map.get(cleaned, cleaned)

# Build mapping from the *rawest* table (person-session)
raw_names = ps["person_name"].astype(str).fillna("")
freq = Counter(raw_names)
unique_raw = pd.Series(sorted(set(raw_names)))
cleaned = unique_raw.apply(final_clean)

def keyize(s: str) -> str:
    s2 = s.lower()
    s2 = re.sub(r"[^a-z]", "", s2)  # letters only for a coarse cluster key
    return s2

map_df = pd.DataFrame({
    "raw_name": unique_raw,
    "clean_name": cleaned,
})
map_df["raw_freq"] = map_df["raw_name"].map(freq)
map_df["cluster_key"] = map_df["clean_name"].apply(keyize)

# choose canonical per cluster (most frequent raw -> its cleaned)
canon_by_key = {}
for key, grp in map_df.groupby("cluster_key"):
    idx = grp["raw_freq"].idxmax()
    canon_by_key[key] = grp.loc[idx, "clean_name"]
map_df["canonical_name"] = map_df["cluster_key"].map(canon_by_key)

# Save mapping for review
map_df.sort_values(["cluster_key","clean_name","raw_freq"], ascending=[True, True, False]).to_csv(MAP_OUT, index=False)
print(f"[OK] Wrote name mapping: {MAP_OUT}")

# Helper to apply mapping (raw->canonical) with fallback to cleaner
raw_to_canon = dict(zip(map_df["raw_name"], map_df["canonical_name"]))

def apply_canonical(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["person_name"] = df["person_name"].map(raw_to_canon).fillna(df["person_name"].apply(final_clean))
    return df

# ========= APPLY TO PERSON-SESSION (and write) =========
ps_clean = apply_canonical(ps)
ps_clean.to_csv(PS_OUT, index=False)
print(f"[OK] Wrote cleaned person-session: {PS_OUT}  (rows={len(ps_clean)})")

# ========= RE-AGG LOGIC (keeps ints for counts, floats for means) =========
def build_agg_dict(df: pd.DataFrame):
    sum_cols = []
    mean_cols = []
    for c in df.columns:
        if c in ["person_name","conference","year","session_id","role_in_session","role_primary","team_ids"]:
            continue
        # counts / sums (discrete events) -> SUM
        if (c.endswith("_count") or c.endswith("_sum_score") or
            c in ["p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
                  "sessions_total","facilitator","member","participant","unknown",
                  "teams_funded","teams_unfunded","teams_total",
                  "role_facilitator","role_nonfacilitator","role_on_team","role_in_funded",
                  "role_member","role_participant"]):
            sum_cols.append(c)
        # means / continuous -> MEAN
        elif (c.endswith("_mean_score") or c.endswith("_mean_score_weighted") or
              c.startswith("ctx_") or c in ["p_speaking_duration_sec"]):
            mean_cols.append(c)
        else:
            # default: if numeric and not a known count, treat as mean to be safe
            if pd.api.types.is_numeric_dtype(df[c]):
                mean_cols.append(c)
    agg = {**{c:"sum" for c in sum_cols}, **{c:"mean" for c in mean_cols}}
    return agg, sum_cols

def cast_int_counts(df: pd.DataFrame, count_cols):
    for c in count_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).round().astype(int)
    return df

# ========= RE-AGG PERSON–YEAR =========
py_clean = apply_canonical(py)
agg_py, sum_cols_py = build_agg_dict(py_clean)
py_final = (py_clean
            .groupby(["person_name","conference","year"], as_index=False)
            .agg(agg_py)
           )
py_final = cast_int_counts(py_final, sum_cols_py + ["year"])
py_final.to_csv(PY_OUT, index=False)
print(f"[OK] Wrote cleaned person–year: {PY_OUT}  (rows={len(py_final)})")

# ========= RE-AGG PERSON-LEVEL =========
p_clean = apply_canonical(p)
agg_p, sum_cols_p = build_agg_dict(p_clean)
p_final = (p_clean
           .groupby(["person_name"], as_index=False)
           .agg(agg_p)
          )
p_final = cast_int_counts(p_final, sum_cols_p)
p_final.to_csv(P_OUT, index=False)
print(f"[OK] Wrote cleaned person-level: {P_OUT}  (rows={len(p_final)})")

# ========= QUICK DIAGNOSTICS =========
sus = map_df[ (map_df["clean_name"].eq("")) | (~map_df["clean_name"].str.contains(r"\s")) ]
if not sus.empty:
    print("\n[Heads-up] Some names look empty or single-token; consider alias_map fixes. Examples:")
    print(sus.sort_values("raw_freq", ascending=False).head(15)[["raw_name","clean_name","canonical_name","raw_freq"]].to_string(index=False))
else:
    print("\n[OK] No obviously empty/single-token names in mapping.")

[OK] Wrote name mapping: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/name_clean_map.csv
[OK] Wrote cleaned person-session: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_session_CLEANED.csv  (rows=2073)
[OK] Wrote cleaned person–year: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year_FIXED_CLEANED.csv  (rows=771)
[OK] Wrote cleaned person-level: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/all_data_df-agg-ppl_FIXED_CLEANED.csv  (rows=637)

[Heads-up] Some names look empty or single-token; consider alias_map fixes. Examples:
                raw_name clean_name canonical_name  raw_freq
    Marie-Claire Arrieta      Marie          Marie         4
            John-Paul Yu       John           John         4
             Gang-yu Liu       Gang           Gang         4
   Anna-Karin Gustavsson       An

## rebuilding the analysis part 

In [11]:
# =========================================
# REBUILD ANALYSIS: Charts + Regressions + ROC (robust to collinearity)
# Requires: pandas, numpy, matplotlib, statsmodels, scikit-learn, xlsxwriter, tabulate
# =========================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.sm_exceptions import PerfectSeparationError
from patsy import dmatrices
import numpy.linalg as npl

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, classification_report
from sklearn.impute import SimpleImputer

# ------------------ CONFIG ------------------
DATA_CSV = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year.csv")
OUT_DIR  = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS     = 5
INCLUDE_CTX  = True   # include ctx_* features along with p_* and ann_* in regressions and classifiers

# ------------------ LOAD ------------------
df = pd.read_csv(DATA_CSV)

# ---- Ensure role flags exist (derive if missing) ----
def ensure_role_flags(df):
    df = df.copy()

    # role_facilitator
    if "role_facilitator" not in df.columns:
        if "facilitator" in df.columns:
            df["role_facilitator"] = (pd.to_numeric(df["facilitator"], errors="coerce").fillna(0) > 0).astype(int)
        elif "role_primary" in df.columns:
            df["role_facilitator"] = (df["role_primary"].astype(str).str.lower() == "facilitator").astype(int)
        else:
            df["role_facilitator"] = 0

    # role_on_team
    if "role_on_team" not in df.columns:
        if "teams_total" in df.columns:
            df["role_on_team"] = (pd.to_numeric(df["teams_total"], errors="coerce").fillna(0) > 0).astype(int)
        elif "member" in df.columns:
            df["role_on_team"] = (pd.to_numeric(df["member"], errors="coerce").fillna(0) > 0).astype(int)
        else:
            df["role_on_team"] = 0

    # role_in_funded
    if "role_in_funded" not in df.columns:
        if "teams_funded" in df.columns:
            df["role_in_funded"] = (pd.to_numeric(df["teams_funded"], errors="coerce").fillna(0) > 0).astype(int)
        else:
            df["role_in_funded"] = 0

    for c in ["role_facilitator","role_on_team","role_in_funded"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df

df = ensure_role_flags(df)

# Which flags actually exist?
role_flags_available = [c for c in ["role_facilitator","role_on_team","role_in_funded"] if c in df.columns]

# ------------------ FEATURE SETS ------------------
# Person-level behavioral (transcript-derived)
p_feats   = [c for c in df.columns if c.startswith("p_") and pd.api.types.is_numeric_dtype(df[c])]
# Annotation categories
ann_feats = [c for c in df.columns if c.startswith("ann_") and pd.api.types.is_numeric_dtype(df[c])]
# Session-level context (if present)
ctx_feats = [c for c in df.columns if c.startswith("ctx_") and pd.api.types.is_numeric_dtype(df[c])]
if not INCLUDE_CTX:
    ctx_feats = []

num_feats = sorted(list(set(p_feats + ann_feats + ctx_feats)))

# Drop zero-variance / all-missing features
good_feats = []
for c in num_feats:
    s = pd.to_numeric(df[c], errors="coerce")
    if s.notna().any() and s.std(skipna=True) > 0:
        good_feats.append(c)
num_feats = sorted(good_feats)

print(f"Using {len(num_feats)} numeric features.")

# ------------------ CHARTS: Helper Plots ------------------
def bar_mean_with_ci(data, by_flag, metric, title, out_png):
    tmp = data[[by_flag, metric]].dropna()
    tmp[by_flag] = tmp[by_flag].astype(int)
    groups = tmp.groupby(by_flag)[metric]
    means = groups.mean()
    ns    = groups.size()
    sds   = groups.std()
    cis = 1.96 * sds / np.sqrt(ns.clip(lower=1))  # 95% CI

    fig, ax = plt.subplots(figsize=(5,4))
    ax.bar(["No","Yes"], [means.get(0, np.nan), means.get(1, np.nan)])
    ax.errorbar([0,1], [means.get(0, np.nan), means.get(1, np.nan)],
                yerr=[cis.get(0,0), cis.get(1,0)], fmt='none', capsize=4)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_xlabel(by_flag)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180)
    plt.close(fig)

def box_by_flag(data, by_flag, metric, title, out_png):
    tmp = data[[by_flag, metric]].dropna()
    tmp[by_flag] = tmp[by_flag].astype(int)
    fig, ax = plt.subplots(figsize=(5,4))
    ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
               labels=["No","Yes"], showfliers=False)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_xlabel(by_flag)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180)
    plt.close(fig)

# Pick a concise subset of metrics to visualize (add more if you like)
bar_metrics = [
    "p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "ann_knowledge_sharing_count","ann_evaluation_practices_count",
    "ann_coordination_and_decision_practices_count","ann_relational_climate_count",
    "ann_participation_dynamics_count"
]
box_metrics = ["p_speaking_duration_sec","p_turns"]

# Make charts for each contrast flag that exists
for flag in role_flags_available:
    for m in bar_metrics:
        if m in df.columns:
            fn = OUT_DIR / f"bar_{m}_{flag}.png"
            bar_mean_with_ci(df, flag, m, f"{m} by {flag}", fn)
    for m in box_metrics:
        if m in df.columns:
            fn = OUT_DIR / f"box_{m}_{flag}.png"
            box_by_flag(df, flag, m, f"{m} by {flag}", fn)

print("[OK] Wrote bar/box charts.")

# ------------------ REGRESSIONS (Logit with FE, full-rank safe) ------------------
def _drop_problem_cols(X_df):
    """Drop zero-variance, duplicate columns; if still rank-deficient, keep a full-rank subset via QR."""
    X = X_df.copy()

    # Drop all-constant / zero-variance
    keep = [c for c in X.columns if X[c].std(ddof=0) > 0]
    X = X[keep]

    # Drop exact-duplicate columns
    seen = {}
    dedup_keep = []
    for c in X.columns:
        key = tuple(np.asarray(X[c]).round(12))
        if key in seen:
            continue
        seen[key] = c
        dedup_keep.append(c)
    X = X[dedup_keep]

    # Full-rank via QR pivot if needed
    A = X.values
    rank = np.linalg.matrix_rank(A)
    if rank == A.shape[1]:
        return X

    Q, R, piv = npl.qr(A, mode='reduced', pivoting=True)
    indep_idx = piv[:rank]
    keep_cols = [X.columns[i] for i in indep_idx]
    return X[keep_cols]

def logit_with_fe(data, target, predictors, fe_conf=True, fe_year=True, add_sess=True):
    """
    Build design via patsy, drop collinear columns, fit Logit.
    If MLE fails (singular/complete separation), fall back to fit_regularized.
    Returns: (data_used, model_or_result, robust_or_none, formula_string)
    """
    # Build formula
    rhs = []
    if add_sess and "sessions_total" in data.columns:
        rhs.append("sessions_total")
    rhs += [f"`{x}`" if (" " in x or "-" in x) else x for x in predictors]
    if fe_conf and "conference" in data.columns:
        rhs.append("C(conference)")
    if fe_year and "year" in data.columns:
        rhs.append("C(year)")
    formula = f"{target} ~ " + " + ".join(rhs)

    # Design matrices
    try:
        y, X = dmatrices(formula, data=data, return_type="dataframe", NA_action="drop")
    except Exception as e:
        print(f"[SKIP] Patsy failed for {target}: {e}")
        return None, None, None, formula

    y_vec = np.asarray(y).ravel().astype(int)
    if np.unique(y_vec).size < 2:
        print(f"[SKIP] {target}: only one class after NA drop.")
        return None, None, None, formula

    # Hold intercept, clean the rest
    const = None
    if "Intercept" in X.columns:
        const = X["Intercept"].copy()
        X = X.drop(columns=["Intercept"])
    X = _drop_problem_cols(X)
    if const is not None:
        X.insert(0, "Intercept", const.values)

    # Fit (request robust SEs directly); fallback to regularized if needed
    try:
        model = sm.Logit(y_vec, X)
        result = model.fit(disp=0, cov_type="HC3")   # <-- robust SEs here
        robust = result                               # already robust
        used = pd.concat([y.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
        return used, result, robust, formula
    except (np.linalg.LinAlgError, PerfectSeparationError) as e:
        print(f"[Info] MLE failed for {target} ({type(e).__name__}); trying regularized fit.")
        model = sm.Logit(y_vec, X)
        result = model.fit_regularized(alpha=1.0, L1_wt=0.5, disp=0)
        robust = None  # robust SEs not available here
        used = pd.concat([y.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
        return used, result, robust, formula

def tidy_from_result(result, robust_or_none, model_label):
    params = (robust_or_none.params if robust_or_none is not None else result.params)
    bse    = (robust_or_none.bse    if robust_or_none is not None else result.bse)
    pvals  = (robust_or_none.pvalues if robust_or_none is not None else result.pvalues)
    conf   = (robust_or_none.conf_int() if robust_or_none is not None else result.conf_int())

    out = pd.DataFrame({
        "predictor": params.index,
        "coef": params.values,
        "std_err": bse.values,
        "p_value": pvals.values,
        "ci_low": conf[0].values,
        "ci_high": conf[1].values,
        "model": model_label,
        "n_obs": int(result.nobs),
        "llf": result.llf,
        "llnull": getattr(result, "llnull", np.nan),
        "pseudo_r2": (1 - (result.llf / result.llnull)) if getattr(result, "llnull", 0) not in (0, np.nan) else np.nan,
        "llr_pvalue": getattr(result, "llr_pvalue", np.nan)
    })
    return out

tidy_list = []
summaries = []

for tgt, desc in [
    ("role_facilitator","Facilitator vs Non-facilitator"),
    ("role_on_team","On-team vs Not-on-team"),
    ("role_in_funded","Funded vs Not-funded")
]:
    if tgt not in role_flags_available:
        continue
    used, res, rob, form = logit_with_fe(df, tgt, predictors=num_feats, fe_conf=True, fe_year=True, add_sess=True)
    if res is None:
        continue

    # summary text
    summ = [f"=== {desc} ({tgt}) ===", f"Formula: {form}", res.summary2().as_text()]
    if rob is None:
        summ.append("\nNote: Regularized fit used (robust SEs not available).")
    else:
        summ.append("\nRobust (HC3) SEs used in tidy table.")
    summaries.append("\n".join(summ))

    tidy_list.append(tidy_from_result(res, rob, desc))

# Save tidy combined and summaries
if tidy_list:
    tidy_df = pd.concat(tidy_list, ignore_index=True)
    tidy_csv = OUT_DIR / "regression_tidy_results.csv"
    tidy_df.to_csv(tidy_csv, index=False)

    with open(OUT_DIR / "regression_model_summaries.txt","w") as f:
        f.write("\n\n".join(summaries))

    # Nicely formatted per-model tables
    def tidy_to_display(df_tidy, model_name):
        sub = df_tidy[df_tidy["model"]==model_name].copy()
        sub.loc[sub["predictor"].str.startswith("C(conference)"), "predictor"] = sub["predictor"].str.replace("C(conference)","Conf_FE", regex=False)
        sub.loc[sub["predictor"].str.startswith("C(year)"), "predictor"] = sub["predictor"].str.replace("C(year)","Year_FE", regex=False)
        sub["Coef (SE)"] = sub["coef"].round(3).astype(str) + " (" + sub["std_err"].round(3).astype(str) + ")"
        sub["p"] = sub["p_value"].apply(lambda x: f"{x:.3g}")
        keep = ["predictor","Coef (SE)","ci_low","ci_high","p","n_obs","pseudo_r2","llr_pvalue"]
        sub = sub[keep].rename(columns={"predictor":"Term","ci_low":"CI 2.5%","ci_high":"CI 97.5%","p":"p-value","n_obs":"N","pseudo_r2":"Pseudo R2","llr_pvalue":"LLR p"})
        return sub

    tables = {}
    for model_name in tidy_df["model"].unique():
        disp = tidy_to_display(tidy_df, model_name)
        safe = "".join(ch for ch in model_name if ch.isalnum() or ch in "_- ").strip().replace(" ","_")
        disp.to_csv(OUT_DIR / f"reg_table_{safe}.csv", index=False)
        try:
            disp.to_markdown(OUT_DIR / f"reg_table_{safe}.md", index=False)
        except Exception:
            pass
        tables[model_name] = disp

    xlsx_path = OUT_DIR / "regression_tables_by_model.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xw:
        for model_name, disp in tables.items():
            sheet = model_name[:31]
            disp.to_excel(xw, index=False, sheet_name=sheet)

    print(f"[OK] Regressions saved: {tidy_csv} and {xlsx_path}")
else:
    print("[SKIP] No regression models were fit (missing targets or single-class after filtering).")

# ------------------ MULTICOLLINEARITY (VIF on key behaviors) ------------------
vif_feats = [c for c in ["p_speaking_duration_sec","p_turns","p_interruptions_made","p_overlaps",
                         "p_screenshare_segments","p_smile_self_total","p_smile_other_total",
                         "p_nods_received"] if c in df.columns]
if vif_feats:
    X_vif = df[vif_feats].fillna(0)
    X_vif = sm.add_constant(X_vif)
    vif_table = pd.DataFrame({
        "feature": X_vif.columns,
        "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    })
    vif_table.to_csv(OUT_DIR / "vif_behaviors.csv", index=False)
    print("[OK] VIF table written.")
else:
    print("[SKIP] No VIF — none of the key behavior features found.")

# ------------------ PREDICTIVE: ROC Curves with behavior-only ------------------
targets = [(t, d) for (t, d) in [
    ("role_facilitator","Facilitator vs Non-facilitator"),
    ("role_on_team","On-team vs Not-on-team"),
    ("role_in_funded","Funded vs Not-funded")
] if t in role_flags_available]

X_cols = num_feats  # p_, ann_, and (ctx_ if INCLUDE_CTX)
if targets and X_cols:
    numeric_transform = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scale", StandardScaler())
    ])
    preprocess = ColumnTransformer(transformers=[("num", numeric_transform, X_cols)], remainder="drop")

    logit = LogisticRegression(
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    )
    clf = Pipeline(steps=[("prep", preprocess), ("clf", logit)])
    cv  = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    summary_rows = []
    for target, desc in targets:
        dsub = df.dropna(subset=[target]).copy()
        if dsub[target].nunique() < 2:
            print(f"[SKIP] ROC for {target}: single class present.")
            continue

        y = dsub[target].astype(int).values
        X = dsub[X_cols]

        # CV probs + ROC
        y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:,1]
        fpr, tpr, _ = roc_curve(y, y_prob)
        roc_auc = auc(fpr, tpr)

        # Classification report at 0.5
        y_hat = (y_prob >= 0.5).astype(int)
        cr = classification_report(y, y_hat, output_dict=True, zero_division=0)
        pd.DataFrame(cr).to_csv(OUT_DIR / f"clf_classification_report_{target}.csv")

        # CV metric summary
        scoring = {"roc_auc": "roc_auc","accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1"}
        cv_res = cross_validate(clf, X, y, cv=cv, scoring=scoring, return_train_score=False, n_jobs=-1)
        row = {"target": target, "description": desc}
        for m in scoring:
            row[f"{m}_mean"] = float(cv_res[f"test_{m}"].mean())
            row[f"{m}_sd"]   = float(cv_res[f"test_{m}"].std())
        summary_rows.append(row)

        # ROC plot
        fig, ax = plt.subplots(figsize=(5,4))
        ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
        ax.plot([0,1],[0,1], linestyle="--")
        ax.set_title(f"ROC — {desc}")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right")
        plt.tight_layout()
        fig.savefig(OUT_DIR / f"clf_roc_{target}.png", dpi=180)
        plt.close(fig)

    if summary_rows:
        summary_df = pd.DataFrame(summary_rows).sort_values("target")
        summary_df.to_csv(OUT_DIR / "clf_results_summary.csv", index=False)
        print("[OK] Predictive ROC + summaries written.")
else:
    print("[SKIP] ROC — no targets available or no numeric features.")

print("\nOutputs written to:", OUT_DIR)

Using 33 numeric features.


/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metri

[OK] Wrote bar/box charts.
[Info] MLE failed for role_facilitator (LinAlgError); trying regularized fit.
[SKIP] role_in_funded: only one class after NA drop.
[OK] Regressions saved: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild/regression_tidy_results.csv and /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild/regression_tables_by_model.xlsx
[OK] VIF table written.
[SKIP] ROC for role_in_funded: single class present.
[OK] Predictive ROC + summaries written.

Outputs written to: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild


## analysis - 3 categorical

In [18]:
# =========================================
# Meeting-level analysis (person-year data)
# - Outlier checks for controls
# - Single-feature OLS (HC3 SE) on num_teams / num_funded_teams
# - Tidy CSVs + Excel
# - Coef barplots with 95% CI
# - Big grid image: hist + scatters for each feature
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------ CONFIG ------------------
DATA_CSV = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year_WITH_OUTCOMES.csv")
OUT_DIR  = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Which outcomes to model (must exist in the CSV)
OUTCOMES = ["num_teams", "num_funded_teams"]

# Which controls to include in every single-feature regression
CONTROLS = ["p_speaking_duration_sec", "p_turns"]  # (Evey’s suggestion)

# Number of bars to show (by smallest p) in coef barplots
TOP_N_FOR_PLOT = 15

RANDOM_STATE = 42

# ------------------ LOAD ------------------
df = pd.read_csv(DATA_CSV)

# Basic hygiene: make sure outcomes exist & are numeric
for c in OUTCOMES:
    if c not in df.columns:
        raise ValueError(f"Outcome '{c}' not found in {DATA_CSV}.")
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Predictors = p_* and ann_* numeric columns
p_feats   = [c for c in df.columns if c.startswith("p_")   and pd.api.types.is_numeric_dtype(df[c])]
ann_feats = [c for c in df.columns if c.startswith("ann_") and pd.api.types.is_numeric_dtype(df[c])]
predictors = sorted(list(set(p_feats + ann_feats)))

# Drop outcomes / predictors that are entirely missing
df = df.copy()
df[predictors + OUTCOMES] = df[predictors + OUTCOMES].apply(pd.to_numeric, errors="coerce")

# ------------------ Quick outlier checks for controls ------------------
def quick_outlier_plots(data: pd.DataFrame, cols: list, out_dir: Path):
    for c in cols:
        if c not in data.columns:
            continue
        s = pd.to_numeric(data[c], errors="coerce")
        fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
        # Histogram (log x if long tail)
        ax[0].hist(s.dropna(), bins=40)
        ax[0].set_title(f"{c} — histogram")
        ax[0].set_xlabel(c); ax[0].set_ylabel("count")
        # Boxplot (shows extreme points)
        ax[1].boxplot(s.dropna(), vert=True, showfliers=True)
        ax[1].set_title(f"{c} — boxplot")
        ax[1].set_ylabel(c)
        plt.tight_layout()
        fig.savefig(out_dir / f"outlier_{c}.png", dpi=180)
        plt.close(fig)

# scatter against outcomes (to see leverage)
def scatter_vs_outcomes(data: pd.DataFrame, cols: list, outcomes: list, out_dir: Path):
    for c in cols:
        if c not in data.columns:
            continue
        for y in outcomes:
            sub = data[[c, y]].dropna()
            if sub.empty: 
                continue
            fig, ax = plt.subplots(figsize=(4.8, 3.6))
            ax.scatter(sub[c], sub[y], s=10, alpha=0.6)
            ax.set_xlabel(c); ax.set_ylabel(y)
            ax.set_title(f"{y} vs {c}")
            plt.tight_layout()
            fig.savefig(out_dir / f"scatter_{y}_vs_{c}.png", dpi=180)
            plt.close(fig)

quick_outlier_plots(df, CONTROLS, OUT_DIR)
scatter_vs_outcomes(df, CONTROLS, OUTCOMES, OUT_DIR)

# ------------------ Utility: single-feature OLS with HC3 robust SE ------------------
def run_single_feature_ols(data: pd.DataFrame, outcome: str, feature: str, controls: list):
    cols_base = [outcome, feature]
    # exclude the focal feature from the controls to avoid duplicates
    controls_eff = [c for c in controls if (c in data.columns and c != feature)]
    cols = cols_base + controls_eff

    d = data[cols].dropna()
    if d[outcome].nunique() <= 1:
        return None  # cannot regress on constant outcome

    y = d[outcome].astype(float).values
    X = d[[feature] + controls_eff].astype(float)
    X = sm.add_constant(X, has_constant="add")

    model = sm.OLS(y, X).fit(cov_type="HC3")  # HC3 robust SE

    # now `feature` is unique in X, so these are scalars
    out = {
        "outcome": outcome,
        "feature": feature,
        "coef": float(model.params[feature]),
        "std_err": float(model.bse[feature]),
        "t": float(model.tvalues[feature]),
        "p_value": float(model.pvalues[feature]),
        "ci_low": float(model.conf_int().loc[feature, 0]),
        "ci_high": float(model.conf_int().loc[feature, 1]),
        "N": int(model.nobs),
        "R2": float(model.rsquared),
        "Adj_R2": float(model.rsquared_adj),
    }
    return out

# ------------------ Run regressions for each outcome ------------------
all_rows = []
for outcome in OUTCOMES:
    for feat in predictors:
        res = run_single_feature_ols(df, outcome, feat, CONTROLS)
        if res is not None:
            all_rows.append(res)

if not all_rows:
    raise RuntimeError("No models were fit — check your outcomes and predictors.")

results_df = pd.DataFrame(all_rows)
results_df = results_df.sort_values(["outcome", "p_value", "feature"]).reset_index(drop=True)

# Write CSVs per outcome + combined + Excel workbook
results_df.to_csv(OUT_DIR / "single_feature_regressions_ALL.csv", index=False)
for outcome in OUTCOMES:
    sub = results_df[results_df["outcome"] == outcome].copy()
    sub.to_csv(OUT_DIR / f"single_feature_regressions_{outcome}.csv", index=False)

with pd.ExcelWriter(OUT_DIR / "single_feature_regressions_by_outcome.xlsx") as xw:
    for outcome in OUTCOMES:
        sub = results_df[results_df["outcome"] == outcome]
        sub.to_excel(xw, sheet_name=outcome[:31], index=False)

print(f"[OK] wrote regression tables to: {OUT_DIR}")

# ------------------ Coefficient barplots with 95% CI ------------------
def coef_barplot(results: pd.DataFrame, outcome: str, topn: int, out_dir: Path):
    sub = results[results["outcome"] == outcome].copy()
    if sub.empty: 
        return
    # Choose topn by smallest p-value
    sub = sub.nsmallest(topn, "p_value")
    sub = sub.sort_values("coef")  # nicest for bars

    y = np.arange(len(sub))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(y, sub["coef"], xerr=1.96*sub["std_err"], alpha=0.8)
    ax.set_yticks(y, labels=sub["feature"])
    ax.set_xlabel("Coefficient (OLS, HC3 SE)")
    ax.set_title(f"Coefficients with 95% CI — {outcome}")
    for i, (coef) in enumerate(sub["coef"].round(3)):
        ax.text(coef, i, f"  b = {coef:.3f}", va="center")
    plt.tight_layout()
    fig.savefig(out_dir / f"coef_bar_{outcome}.png", dpi=200)
    plt.close(fig)

for outcome in OUTCOMES:
    coef_barplot(results_df, outcome, TOP_N_FOR_PLOT, OUT_DIR)

# ------------------ Big “feature grid” image (hist + two scatters) ------------------
def feature_grid_plot(data: pd.DataFrame, feats: list, outcomes: list, out_dir: Path, per_page: int = 12):
    """
    For each feature: col0 = histogram, col1 = scatter vs outcomes[0], col2 = scatter vs outcomes[1]
    Paginates across multiple JPEGs if needed.
    """
    feats = [f for f in feats if f in data.columns]
    pages = math.ceil(len(feats) / per_page)
    for page in range(pages):
        chunk = feats[page*per_page : (page+1)*per_page]
        nrows = len(chunk)
        ncols = 3
        fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2*nrows), squeeze=False)
        for r, f in enumerate(chunk):
            s = pd.to_numeric(data[f], errors="coerce").dropna()
            # 0) histogram
            axes[r,0].hist(s, bins=40)
            axes[r,0].set_title(f"{f} — distribution")
            axes[r,0].set_ylabel("count")
            # 1) scatter vs first outcome
            if outcomes[0] in data.columns:
                sub = data[[f, outcomes[0]]].dropna()
                axes[r,1].scatter(sub[f], sub[outcomes[0]], s=8, alpha=0.6, color="C0")
                axes[r,1].set_title(f"{outcomes[0]} vs {f}")
                axes[r,1].set_ylabel(outcomes[0])
            # 2) scatter vs second outcome
            if len(outcomes) > 1 and outcomes[1] in data.columns:
                sub = data[[f, outcomes[1]]].dropna()
                axes[r,2].scatter(sub[f], sub[outcomes[1]], s=8, alpha=0.6, color="C3")
                axes[r,2].set_title(f"{outcomes[1]} vs {f}")
                axes[r,2].set_ylabel(outcomes[1])

            for c in range(ncols):
                axes[r,c].set_xlabel(f)

        plt.tight_layout(h_pad=1.2, w_pad=1.2)
        out_path = out_dir / f"feature_analysis_grid_p{page+1}.jpeg"
        fig.savefig(out_path, dpi=160, pil_kwargs={"quality": 90})
        plt.close(fig)

# Build the grid over all predictors
feature_grid_plot(df, predictors, OUTCOMES, OUT_DIR, per_page=12)

# ------------------ VIF for controls (sanity check) ------------------
vif_feats = [c for c in CONTROLS if c in df.columns]
if len(vif_feats) >= 2:
    X_vif = df[vif_feats].dropna().astype(float)
    X_vif = sm.add_constant(X_vif)
    vif_tbl = pd.DataFrame({
        "feature": X_vif.columns,
        "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    })
    vif_tbl.to_csv(OUT_DIR / "vif_controls.csv", index=False)
    print("[OK] VIF written:", OUT_DIR / "vif_controls.csv")
else:
    print("[SKIP] VIF — need at least 2 control variables present.")

print("\nDone. Outputs in:", OUT_DIR)

[OK] wrote regression tables to: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level
[OK] VIF written: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/vif_controls.csv

Done. Outputs in: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level


### fixing the graph visually (so names do not run onto each other on title)

[OK] Wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/coef_bar_num_teams_clean.png
[OK] Wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/coef_bar_num_funded_clean.png
[OK] Wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/coef_bar_facilitator_logit_clean.png
[OK] Wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/feature_analysis_grid_clean.png

All set! Figure pack at: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/FigurePack.pdf
Cleaned figures in: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean


In [22]:
import matplotlib.pyplot as plt
import textwrap
from matplotlib.backends.backend_pdf import PdfPages

def safe_title(text, width=25, fontsize=7):
    """Wrap long titles and shrink fontsize if needed."""
    return "\n".join(textwrap.wrap(text, width)), fontsize

# Collect features (p_* and ann_*)
features_to_plot = [c for c in df.columns 
                    if (c.startswith("ann_") or c.startswith("p_")) 
                    and pd.api.types.is_numeric_dtype(df[c])]

pdf_out = OUT_DIR / "feature_analysis_all.pdf"
with PdfPages(pdf_out) as pdf:
    # batch 9 features per page (3 rows × 3 cols)
    batch_size = 9
    for i in range(0, len(features_to_plot), batch_size):
        batch = features_to_plot[i:i+batch_size]

        fig, axes = plt.subplots(3, 3, figsize=(12, 12))
        axes = axes.flatten()

        for ax, feat in zip(axes, batch):
            if feat not in df.columns:
                continue

            # Example: histogram for distribution
            ax.hist(df[feat].dropna(), bins=30, color="steelblue", alpha=0.7)

            # safe wrapped title
            t, fs = safe_title(f"{feat} — distribution")
            ax.set_title(t, fontsize=fs)

        # Remove any unused subplots in the grid
        for ax in axes[len(batch):]:
            ax.axis("off")

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"[OK] Combined plots saved to: {pdf_out}")

[OK] Combined plots saved to: outputs_meeting_level_cleaned_titles/feature_analysis_all.pdf
